# Обучение и проверка (Colab, GPU)

Тонкий блокнот: весь код в пакете `anemia/` и в `scripts/` репозитория. Запускайте ячейки по порядку.

**Данные и модели без Google Диска.** Ячейка 2 сама качает оба датасета с Kaggle (без аккаунта) и сверяет каждый файл
по sha1. Ячейка Я1 берёт готовую модель взрослых из GitHub Release репозитория (сверка sha256) и обучает только то,
чего нет. Всё лежит в `/content/anemia` до конца сеанса Colab; в новом сеансе ячейки 2 и Я1 скачивают заново
(≈1–3 мин), а не обучают. В репозиторий данные не попадают.

**Обновление кода.** Ячейка 1 при каждом запуске подтягивает `origin/main` (галочка «Обновить_код»), печатает новые коммиты и статус тестов GitHub. Правки, сделанные прямо в `/content/repo`, не теряются: они уходят в `git stash`. Код новых коммитов выполняется только когда вы запускаете ячейки 2–3.

In [ ]:
#@title Ячейка 1 — репозиторий, зависимости, хранилище
#@markdown Google Диск сохраняет данные и модели между сессиями. Если Диск не подключается — снимите галочку:
#@markdown всё будет в /content (стирается при закрытии сессии, данные скачаются заново).
Использовать_Google_Диск = False  #@param {type:"boolean"}
#@markdown **Код из GitHub.** Галочка «Обновить_код» подтягивает свежую версию ветки и печатает список новых коммитов;
#@markdown без галочки остаётся уже скачанная версия. «Коммит» — закрепить конкретный коммит (пусто = последний в ветке).
Обновить_код = True  #@param {type:"boolean"}
Ветка = 'main'  #@param {type:"string"}
Коммит = ''  #@param {type:"string"}
import os, subprocess, sys, time, json, shutil, hashlib, urllib.request
T0 = time.time()
def step(name):
    print(f'[{time.time() - T0:5.1f} с] {name}', flush=True)
os.environ['GIT_TERMINAL_PROMPT'] = '0'   # git никогда не ждёт ввода логина/пароля: при ошибке сразу падает
REPO = 'zhansaya-afk/jjghg'
REPO_DIR = '/content/repo'
BRANCH = Ветка.strip() or 'main'
BASE = '/content/anemia'
if Использовать_Google_Диск:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        BASE = '/content/drive/MyDrive/anemia'
    except Exception as e:
        print('⚠️ Google Диск не подключился, работаю без него (данные в /content/anemia):', str(e)[:150])
os.makedirs(BASE, exist_ok=True)
os.environ['ANEMIA_BASE'] = BASE
os.environ['ANEMIA_DATA'] = f'{BASE}/data'
os.environ['ANEMIA_MODELS'] = f'{BASE}/models'

HIST = 'results/metrics_history.csv'   # история метрик: её дописывает ячейка 3, при обновлении кода не прячем


def history_extra(head_text, local_text):
    """Строки, дописанные к истории после коммита (local = HEAD + новые строки) → текст, иначе None."""
    if head_text and head_text.endswith('\n') and local_text.startswith(head_text):
        return local_text[len(head_text):]
    return None


def run(cmd, timeout, what, **kw):
    """subprocess.run с таймаутом: нет сети или GitHub не отвечает → понятная ошибка, а не вечное ожидание."""
    try:
        return subprocess.run(cmd, timeout=timeout, **kw)
    except subprocess.TimeoutExpired:
        raise RuntimeError(f'{what}: нет ответа {timeout} с — проверьте интернет и github.com, затем запустите '
                           'ячейку снова (или снимите «Обновить_код», чтобы работать на уже скачанной версии)') from None


def git(*args):
    r = run(['git', '-C', REPO_DIR, *args], 120, f'git {args[0]}', capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f'git {" ".join(args)}: {r.stderr.strip()[:300]}')
    return r.stdout.strip()

def token_url():
    """Токен из Colab Secrets (GITHUB_TOKEN) — только если без него GitHub не отдал репозиторий (приватный).
    userdata.get может открыть окно «разрешить доступ к секрету» и ждать нажатия, поэтому сначала — без токена."""
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        token = None
    return f'https://{token}@github.com/{REPO}.git' if token else None


url = f'https://github.com/{REPO}.git'   # репозиторий публичный: токен не нужен
if not os.path.isdir(f'{REPO_DIR}/.git'):
    step('клонирую репозиторий...')
    if run(['git', 'clone', '-q', '-b', BRANCH, url, REPO_DIR], 300, 'git clone').returncode:
        url = token_url()
        if not url or run(['git', 'clone', '-q', '-b', BRANCH, url, REPO_DIR], 300, 'git clone').returncode:
            raise RuntimeError(f'не удалось скачать {REPO} (ветка {BRANCH}): проверьте интернет, имя ветки, '
                               'для приватного репозитория — секрет GITHUB_TOKEN')
    if Коммит.strip():
        git('reset', '-q', '--hard', Коммит.strip())
elif Обновить_код:
    step('подтягиваю origin/' + BRANCH + '...')
    try:
        git('fetch', '-q', 'origin', BRANCH)
    except RuntimeError:
        if not token_url():
            raise
        git('fetch', '-q', token_url(), f'+refs/heads/{BRANCH}:refs/remotes/origin/{BRANCH}')
    target = git('rev-parse', '--verify', (Коммит.strip() or f'origin/{BRANCH}') + '^{commit}')
    stamp = time.strftime('%Y%m%d-%H%M%S')
    # новые строки истории метрик: резервная копия в хранилище, файл — к версии HEAD, строки — обратно после обновления
    hist_rows, hist_path = None, f'{REPO_DIR}/{HIST}'
    hist_backup = f'{BASE}/metrics_history_local-{stamp}.csv'
    if os.path.isfile(hist_path) and git('status', '--porcelain', '--', HIST):   # нет файла → как раньше, без копии
        shutil.copy2(hist_path, hist_backup)
        head = subprocess.run(['git', '-C', REPO_DIR, 'show', f'HEAD:{HIST}'], capture_output=True, text=True).stdout
        hist_rows = history_extra(head, open(hist_path, encoding='utf-8').read())
        if hist_rows is None:
            print(f'⚠️ {HIST} изменён не дописыванием строк — уйдёт в stash; копия: {hist_backup}')
        else:
            git('checkout', 'HEAD', '--', HIST)   # и рабочая копия, и индекс: в stash строк не будет, дублей нет
    # ничего не теряем: правки в /content/repo уходят в stash, свои коммиты — в ветку colab-backup-*
    if git('status', '--porcelain'):
        git('stash', 'push', '-u', '-m', f'colab-{stamp}')
        print(f'⚠️ Локальные правки в {REPO_DIR} сохранены: git -C {REPO_DIR} stash list')
    if git('log', '--oneline', 'HEAD', '--not', target, f'origin/{BRANCH}'):
        git('branch', f'colab-backup-{stamp}')
        print(f'⚠️ Коммиты, которых нет в {BRANCH}, сохранены в ветке colab-backup-{stamp}')
    incoming = git('log', '--oneline', '--no-decorate', f'HEAD..{target}')
    git('reset', '-q', '--hard', target)
    if hist_rows and not os.path.isfile(hist_path):
        print(f'⚠️ В новой версии нет {HIST}: ваши строки не дописаны, они в {hist_backup}')
    elif hist_rows:
        with open(hist_path, encoding='utf-8') as f:
            new_text = f.read()
        with open(hist_backup, encoding='utf-8') as f:
            same_header = new_text.split('\n', 1)[0] == f.readline().rstrip('\n')
        if same_header:
            with open(hist_path, 'a', encoding='utf-8') as f:
                f.write(('' if new_text.endswith('\n') else '\n') + hist_rows)   # строки не склеиваются
            print(f'Строки истории метрик, дописанные в Colab, сохранены в {HIST}: {hist_rows.count(chr(10))}')
        else:
            print(f'⚠️ У {HIST} в новой версии другие колонки: ваши строки не дописаны, они в {hist_backup}')
    if Коммит.strip():
        print('Закреплён коммит', Коммит.strip())
    print('Новые коммиты:\n' + incoming if incoming else 'Новых коммитов нет.')
else:
    print('Обновление выключено: работаю на уже скачанной версии.')
os.chdir(REPO_DIR)
# pip install только при первом запуске в этой среде или если изменились зависимости (pyproject.toml):
# повторная установка каждый раз собирает пакет заново и проверяет все зависимости — это минуты
deps = hashlib.sha1(open(f'{REPO_DIR}/pyproject.toml', 'rb').read()).hexdigest()
mark = '/content/.anemia_pip_' + deps
import importlib.util
spec = importlib.util.find_spec('anemia')
installed = spec is not None and os.path.realpath(spec.origin or '').startswith(os.path.realpath(REPO_DIR))
if os.path.exists(mark) and installed:   # пакет стоит из этой папки и зависимости не менялись
    step('зависимости уже установлены в этой среде — пропускаю pip')
else:
    step('устанавливаю зависимости (только первый раз в новой среде, ≈1–2 мин)...')
    if run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[data,dev]'], 900, 'pip install').returncode:
        raise RuntimeError('pip install не прошёл — вывод выше; без зависимостей дальше запускать нельзя')
    open(mark, 'w').close()
sha = git('rev-parse', 'HEAD')
print('Хранилище:', BASE)
print('Коммит:', git('log', '-1', '--format=%h %s (%cd)', '--date=short'))
# статус тестов GitHub Actions для этого коммита (без токена, только для публичного репозитория)
try:
    with urllib.request.urlopen(f'https://api.github.com/repos/{REPO}/commits/{sha}/check-runs', timeout=10) as r:
        runs = json.load(r).get('check_runs', [])
    tests = [c for c in runs if c['name'].startswith('pytest')]
    if not tests:
        print('Тесты GitHub: нет данных для этого коммита')
    elif all(c['conclusion'] == 'success' for c in tests):
        print('Тесты GitHub: ✅ пройдены')
    elif any(c['status'] != 'completed' for c in tests):
        print('Тесты GitHub: ⏳ ещё идут')
    else:
        print('Тесты GitHub: ❌ упали — сообщите Claude, прежде чем запускать дальше')
except Exception as e:
    print('Тесты GitHub: статус не получен:', str(e)[:80])
if 'anemia' in sys.modules:
    print('ℹ️ Пакет anemia уже был импортирован: чтобы взять новый код, «Среда выполнения → Перезапустить сеанс» и снова ячейка 1.')
step('готово')

In [ ]:
#@title Ячейка 2 — данные: скачать (один раз за сеанс) и собрать manifest.csv
# Eyes-defy-anemia качается с Kaggle без аккаунта; каждый файл сверяется по sha1 (scripts/eyes_defy.sha1).
# CP-AnemiC исключён из проекта (2026-10-19): не скачивается; старая папка cp_anemic/ в манифест не попадает.
import os
D = os.environ['ANEMIA_DATA']
if not os.path.exists(f'{D}/eyes_defy_anemia/labels.csv'):
    !python scripts/fetch_data.py --eyes-defy
else:   # файлы уже есть: только пересобрать labels.csv текущим кодом (2026-10-20: опечатки в именах масок), ~1 с
    !python scripts/fetch_data.py --relabel
# 83 телефонных фото с масками конъюнктивы (Kaggle, 7 МБ): учат сегментатор века (не модель риска), anemia/external.py
if not os.path.isdir(f'{D}/external/phone_conj_masks/raw'):
    !python scripts/fetch_data.py --external phone_conj_masks
!python scripts/prepare_data.py
!python -m pytest -q


In [ ]:
#@title Ячейка 3 — честный baseline (этап 1): сегментация v11 + Hb по цвету, все датасеты
# ≈10–20 мин на CPU/GPU Colab. Пришлите Claude весь вывод ячейки: таблицу и строки CSV в конце.
import os, pandas as pd
before = len(pd.read_csv('results/metrics_history.csv')) if os.path.exists('results/metrics_history.csv') else 0
!python scripts/evaluate.py --tag baseline_v11
h = pd.read_csv('results/metrics_history.csv')
new = h.iloc[before:]
new.to_csv(f"{os.environ['ANEMIA_BASE']}/metrics_history_colab.csv", mode='a', index=False,
           header=not os.path.exists(f"{os.environ['ANEMIA_BASE']}/metrics_history_colab.csv"))
print('\n--- строки для results/metrics_history.csv ---')
print(new.to_csv(index=False))

## Ядро HemaVision: фото → риск анемии

Нужны ячейки 1–2 (код и `manifest.csv`). **Ячейка Я1** один раз: честная проверка на фолдах по пациентам
(взрослые Eyes-defy; детей до 5 лет система не оценивает) и обучение итоговой модели → `models/hemavision_core.joblib`.
**Ячейка Я2** — сколько угодно раз: одно фото → риск. **Ячейка Я3** — готовый анализ крови, отдельно от фото.

In [ ]:
#@title Ячейка Я1 — модель ядра: готовая или обучение { display-mode: "form" }
#@markdown Обучение только там, где нет совместимой модели: модель взрослых скачивается из GitHub Release
#@markdown (сверка sha256; подходит, если совпали код признаков и версия scikit-learn). Без готовой модели взрослых — обучение ≈10–20 мин (признаки 216 снимков).
#@markdown Проверка точности (CV по пациентам, ≈15 мин) — только по галочке.
Проверить_точность = False  #@param {type:"boolean"}
Обучить_заново = False  #@param {type:"boolean"}
import os
!python scripts/train_core.py {'--force' if Обучить_заново else ''}
if Проверить_точность:
    !python scripts/evaluate.py --task core --tag core
print('Модель:', os.path.join(os.environ['ANEMIA_MODELS'], 'hemavision_core.joblib'))


In [ ]:
#@title Ячейка Я2 — одно фото → оценка риска { display-mode: "form" }
#@markdown Фото: глаз с оттянутым нижним веком — крупно, лицо целиком или глаз в части кадра.
#@markdown Возраст младше 5 лет — отказ от оценки (модели для детей нет). Беременность «Не указано» — статус неизвестен.
#@markdown Путь пустой — откроется окно загрузки файла.
Путь_к_фото = ''  #@param {type:"string"}
Пол = 'Женский'  #@param ["Женский", "Мужской", "Не указан"]
Возраст = 30  #@param {type:"number"}
Беременность = 'Не указано'  #@param ["Не указано", "Да", "Нет"]
import subprocess, sys
import matplotlib.pyplot as plt
try:                                       # MediaPipe находит глаз на лице; ставится один раз, без зависимостей
    import mediapipe  # noqa: F401           # (как в ноутбуке v12: не трогает numpy/opencv Colab)
except ImportError:
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'mediapipe'], timeout=300)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'flatbuffers', 'absl-py'], timeout=300)
    except subprocess.TimeoutExpired:
        print('MediaPipe не установился за 5 мин: глаз ищется без поиска лица')
from anemia.predict import analyze_photo, format_result, overlay
src = Путь_к_фото.strip()
if not src:
    from google.colab import files
    up = files.upload()
    src = next(iter(up.values()))          # байты файла: analyze_photo принимает путь, байты, PIL или массив
r = analyze_photo(src, sex={'Женский': 'F', 'Мужской': 'M'}.get(Пол), age=Возраст,
                 pregnant={'Да': True, 'Нет': False}.get(Беременность))
vis = overlay(r)
if vis is not None:
    plt.figure(figsize=(5, 4)); plt.imshow(vis); plt.axis('off')
    plt.title('зелёный — конъюнктива, голубой — склера', fontsize=9); plt.show()
print(format_result(r))

In [ ]:
#@title Ячейка Я3 — готовый анализ крови (отдельно от фото) { display-mode: "form" }
Hb = '12.5'  #@param {type:"string"}
Единица = 'г/дл'  #@param ["г/л", "г/дл", "ммоль/л"]
Пол_анализ = 'Женский'  #@param ["Женский", "Мужской", "Не указан"]
Возраст_анализ = 30  #@param {type:"number"}
Беременность_анализ = 'Не указано'  #@param ["Не указано", "Да", "Нет"]
from anemia.labs import lab_hb_report
lab = lab_hb_report(Hb, Единица, sex={'Женский': 'F', 'Мужской': 'M'}.get(Пол_анализ), age=Возраст_анализ,
                    pregnant={'Да': True, 'Нет': False}.get(Беременность_анализ))
print(lab['text'])

In [ ]:
#@title Ячейка Я4 — внешние наборы Kaggle без Hb: проверка модели { display-mode: "form" }
#@markdown Нужны ячейки 1, 2 и Я1. Докачивает 2 набора (~190 МБ, anemia/external.py), копии Eyes-defy пропускает.
#@markdown Модель риска на них НЕ обучается (Hb нет): считается, сколько фото принято и почему отказ, и «анемия / нет»
#@markdown против метки набора (по сериям кадров). IoU сегментатора на телефонных фото — в evaluate.py --task seg_ext
#@markdown (эти фото уже в обучении сегментатора Я1). ≈5–10 мин.
import os
D = os.environ['ANEMIA_DATA']
!python scripts/fetch_data.py --external     # скачивает только недостающие наборы, индекс собирает заново
!python scripts/evaluate_external.py
